# UGANDA CHRISTIAN UNIVERSITY
## AI Music Studio 3.0 - Compose, Continue, Arrange, Present
**Artificial Intelligence Basics | Tutorials 1–16 | Groups of five | Four-day take-home project | 100 marks**

### Why are we doing a music project in Artificial Intelligence?

In your future career as a **Computer Science, Data Science or Electronics** graduate, you will not always be asked to build systems in areas you already understand. You might be assigned to develop an AI application for a bank, a hospital, a sports organisation, a factory or a music company. You may know how to program or train a model, but know very little about financial transactions, medical measurements, sports performance or musical notes. **That is a normal part of real-world computing work.**

Your first responsibility is not to start writing code. It is to **understand the problem and the data**. What does one record represent? What do the values mean? What patterns might matter? What is the system expected to predict or produce? How will you know whether its output is useful or incorrect? Often, you will need to ask questions, consult documentation, investigate unfamiliar terms and experiment before you can build a useful solution.

This assignment places you in exactly that situation. You are being asked to work with **music**, even if you have never studied music theory or played an instrument. You do not need to become a musician. You will learn just enough about notes, rhythm and musical sequences to represent music as data, train an **LSTM neural network** to learn patterns, and investigate the music it produces. You will then combine what the AI generates with ordinary audio-processing and programming techniques to produce and evaluate a remix.

**The real goal is bigger than music:** practise approaching an unfamiliar domain, understanding its data, selecting and applying an AI method, testing the results, recognising limitations and explaining your decisions. These are skills you can transfer to finance, agriculture, healthcare, sports, engineering and many other fields. You may use AI assistants and online resources to support the work, but your group is responsible for understanding and verifying what the system actually does.

**As you work, keep asking:** *What is the input? What does it mean? What is the AI learning? What is the output? How can we tell if it is useful?*

**Core task:** Train a small LSTM to predict symbolic music events, run controlled experiments, compose your own opening, generate music from it, then arrange the result into a **40–55-second multitrack piece** with melody, bass, chords and percussion.

### Submission: each group submits
1. One completed group report in Word or PDF (5–7 pages excluding evidence), with the group representative's access number, all five members, variant ID, tables, graphs, and links to sound files.
2. The executed `.ipynb` notebook, reproducible from top to bottom.
3. A zipped `ai_music_outputs` folder containing raw AI melody, three finished compositions, the personal-motif remix, MIDI files, and figures.
4. A **five-minute** class demo: play the raw LSTM output and a finished arrangement; explain what came from AI versus arrangement rules.

**Permitted:** AI coding assistants and online references. **Required:** real experimental results, interpretation, original personal motif, clear disclosure of AI-assisted code and music sources. Do not represent auto-generated accompaniment as LSTM output. **Music sources:** use your own, appropriately licensed or public-domain melodies. Don't upload commercial songs without permission.

### Four-day teaching plan
- **Day 1:** Music as data, next-event baselines, the LSTM memory mechanism; train first model.
- **Day 2:** Run three model/sampling experiments, inspect metrics and listen critically.
- **Day 3:** Build/upload a personal MIDI motif; generate continuation; produce arranged music.
- **Day 4:** Compare raw and arranged audio, improve arrangement, write report, rehearse presentation.

**Important:** The bundled pattern data are synthetic educational exercises, not authentic style corpora. Results can be repetitive. A better-sounding arrangement does not demonstrate a better language model.


**New for Studio 3.0:** bring an instrumental MP3/WAV/FLAC, inspect its waveform, estimate beats and musical key, compose an LSTM lead, align it to the accompaniment, and audition three remixes. **No upload?** The notebook creates its own fully playable backing track so everyone can complete every task.

**Important:** LSTM predictions generate symbolic lead events. Beat tracking, scale conversion, synthesis and mixing are *other algorithms*, not extra outputs learned by the LSTM. Short audio excerpts chosen for presentations must be original, public-domain or appropriately licensed.


## Part 1 - Identify your group (5 minutes)
Enter **one representative's access number**, and keep it fixed. A deterministic hash selects the group’s sequence family and experimental variant. Group variants may overlap unless representative access numbers are preregistered. Note the assigned `VARIANT_ID` in your report.

**Note:** Variants are reproducible and can differ substantially, but different access numbers are not guaranteed to receive different variant IDs.


In [ ]:
# GROUP INPUT: EDIT THIS CELL ONLY
ACCESS_NUMBER = "B35448"  # group representative access number
GROUP_MEMBERS = ["Member 1", "Member 2", "Member 3", "Member 4", "Member 5"]
assert ACCESS_NUMBER.strip() and ACCESS_NUMBER != "YOUR_ACCESS_NUMBER", "Enter your access number"
import hashlib, random, pathlib, time, copy, math, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from IPython.display import display, Audio
from scipy.io import wavfile
try:
    import pretty_midi
except ImportError:
    %pip -q install pretty_midi
    import pretty_midi
# Stable per-group randomness; group access number is normalized.
GROUP_KEY = ACCESS_NUMBER.strip().upper()
SEED = int(hashlib.sha256(("UCU-AI-MUSIC-V2:"+GROUP_KEY).encode()).hexdigest()[:8], 16)
VARIANT = SEED % 32
FAMILY = ["Piano phrases", "Percussion grooves", "Electronic motifs", "Folk-style melodies"][VARIANT // 8]
VARIANT_IN_FAMILY = VARIANT % 8 + 1
VARIANT_ID = f"M-{VARIANT+1:02d}"
random.seed(SEED); np.random.seed(SEED % (2**32)); torch.manual_seed(SEED); torch.set_num_threads(2)
OUT = pathlib.Path('/content/ai_music_outputs' if pathlib.Path('/content').exists() else 'ai_music_outputs')
OUT.mkdir(parents=True, exist_ok=True)
print(f"GROUP: {GROUP_KEY} | VARIANT: {VARIANT_ID} | FAMILY: {FAMILY} | pattern set {VARIANT_IN_FAMILY} | seed: {SEED}")
print('Members:', ', '.join(GROUP_MEMBERS))
print('Save this line and include the variant ID in your report.')

## Part 2 - Play a note, then play a sequence (30–45 minutes)
**Music does not enter the LSTM as sound.** Here each integer token means a pitch or rest; drum tokens select kick, snare and hi-hat. Equal time steps make the maths manageable. Observe the piano-roll plot. Edit one token and replay. Explain which property changed. The vocabulary size is the number of event types including rest.

**Think:** A chord is simultaneous notes; this first model predicts only one symbolic event per time step. We later add chord and bass instruments using separate *rule-based arrangement code*, not a second AI model.


### Beginner guide: audio versus musical events
A sound recording is a long list of pressure measurements. A typical second of audio contains thousands of samples; these samples describe *sound*, not individual musical notes. A MIDI event is much smaller: it can say 'play MIDI pitch 64 at this time for this duration'. The LSTM here learns from events, not from a raw MP3 waveform.

**Why represent notes as tokens?** A token is an agreed code for an event, exactly as introductory NLP converts pieces of text into tokens. The model uses an embedding layer to turn each token index into a short learned vector. These vectors are numerical features the LSTM can process. A rest is also an event: silence has musical meaning.

**Listen and reason:** When you change one event to 0, which sound disappears? How would a drum pattern differ from a pitch sequence? You are not required to explain each line of the synthesiser code; identify what enters and leaves the function.

**Audio playback:** Several code cells display a small player with a **▶ button** and a timeline. Run a cell first; then click Play in its output. Audio does not usually begin automatically. Use headphones and adjust volume.


In [ ]:
# Core symbolic datasets: patterned but different for each registered seed.
PITCH_CLASSES = {
 "Piano phrases": [60,62,64,65,67,69,71,72],
 "Electronic motifs": [60,63,65,67,70,72,74,75],
 "Folk-style melodies": [57,59,60,62,64,67,69,72],
 "Percussion grooves": [1,2,3,4],
}
VOCAB = [0] + PITCH_CLASSES[FAMILY]
TOK = {p:i for i,p in enumerate(VOCAB)}; REVTOK = {i:p for p,i in TOK.items()}
RNG = np.random.default_rng(SEED)
# Eight characteristic rhythmic/melodic motifs per family; variations produce different phrase orders.
patterns=[]
for j in range(8):
    if FAMILY == "Percussion grooves":
        p=[1,3,2,3,1,3,2,3]
        p[(j+2)%8] = 4 if j%2 else 0
        if j%3 == 0: p[(j+5)%8] = 1
    else:
        notes=PITCH_CLASSES[FAMILY]
        offsets=[0,2,4,2,1,3,5,3] if FAMILY!="Electronic motifs" else [0,3,5,2,4,1,6,2]
        p=[notes[(k+j+VARIANT_IN_FAMILY)%len(notes)] for k in offsets]
        if j%3==0: p[5]=0
    patterns.append(p)
print('Vocabulary (0 = rest):', VOCAB)
print('One phrase:', patterns[0]); print('Eight phrase templates, 8 steps each.')

In [ ]:
def render_events(events, family=FAMILY, rate=16000, step_sec=.24):
    # Synthetic single-line tones/noise for simple WAV playback without SoundFonts.
    length = int(rate*step_sec); y=np.zeros(length*len(events), dtype=np.float32)
    for i,e in enumerate(events):
        if e==0: continue
        t=np.arange(length)/rate
        if family=="Percussion grooves":
            if e==1: wave=np.sin(2*np.pi*(70*t-35*t*t))*np.exp(-24*t)
            elif e==2: wave=RNG.normal(0,1,length)*np.exp(-27*t)
            elif e==3: wave=RNG.normal(0,1,length)*np.exp(-65*t)
            else: wave=RNG.normal(0,1,length)*np.exp(-42*t)
        else:
            freq=440*2**((e-69)/12)
            wave=(np.sin(2*np.pi*freq*t)+.2*np.sin(2*np.pi*2*freq*t))*np.exp(-7*t)
        y[i*length:(i+1)*length]+=(.27*wave).astype(np.float32)
    return np.int16(np.clip(y,-.98,.98)*32767),rate

def save_audio(events, filename, family=FAMILY):
    samples,sr=render_events(events, family); wavfile.write(str(OUT/filename),sr,samples)
    return OUT/filename

def plot_roll(events,title='Event sequence'):
    fig,ax=plt.subplots(figsize=(11,2.6))
    for i,p in enumerate(events):
        if p!=0: ax.broken_barh([(i,.9)], (p-.4,.8), facecolors='tab:blue')
    ax.set_title(title); ax.set_xlabel('Time step (each = half beat)'); ax.set_ylabel('MIDI pitch / drum token');plt.tight_layout();plt.show()
example=patterns[0]*2
plot_roll(example,'Part 2: example musical events')
print('Original:', example)
display(Audio(data=render_events(example)[0],rate=16000))
# ACTIVITY: change an event in `edited` and replay it.
edited=example.copy(); edited[3]=0
print('Edited:',edited)
display(Audio(data=render_events(edited)[0],rate=16000))

**Report task 1 (10 marks):** Identify your musical family and variant; explain what a musical event, rest, pitch and time step mean in this notebook. Describe two audible or visible differences after editing a note. For percussion groups, a token identifies a drum event rather than a pitched note.

## Part 3. Baseline: predict the next event without neural networks
A simple **first-order transition model** estimates the frequency with which one event follows another. It remembers only the most recent event. It provides a baseline against which we can evaluate the LSTM. The dataset below has repeated motifs with moderate variation; strong numbers on this synthetic task should not be mistaken for performance on commercial music.

### Why build a non-neural baseline?
Before claiming that a deep-learning model is useful, compare it with a simple predictor. A first-order transition model counts pairs of consecutive events. If event G is often followed by C, the model will place more probability on C. It cannot consider the last eight notes together: it only checks the immediate previous event.

**Cross-entropy** measures how surprised the predictor is by the correct next event. If it assigns tiny probability to the correct note, loss is high. If it assigns high probability, loss is low. We average this quantity over held-out tracks. Lower is better for *next-event prediction*, but does not prove that generated music sounds beautiful.

**Checkpoint:** Explain what the baseline 'remembers'. Why do we split entire tracks between training and validation instead of mixing almost identical windows across both?


In [ ]:
# Split by complete sequences (no near-identical augmented copies crossing train/validation).
# Synthetic sequence pieces, generated separately with group-specific phrase selections.
def create_sequence(rng, n_phrases=12):
    out=[]
    prev=int(rng.integers(0,8))
    for _ in range(n_phrases):
        j=(prev+int(rng.choice([0,1,2,3],p=[.32,.37,.21,.10])))%8
        phrase=patterns[j].copy()
        if rng.random()<.18:
            idx=int(rng.integers(0,8)); phrase[idx]=VOCAB[int(rng.integers(0,len(VOCAB)))]
        out+=phrase;prev=j
    return out
train_tracks=[create_sequence(RNG) for _ in range(65)]
val_tracks=[create_sequence(RNG) for _ in range(14)]
print('Train tracks:',len(train_tracks),'Validation tracks:',len(val_tracks), 'events/track:',len(train_tracks[0]))
counts=np.ones((len(VOCAB),len(VOCAB)),dtype=np.float64)*.25
for track in train_tracks:
    for a,b in zip(track[:-1],track[1:]):counts[TOK[a],TOK[b]]+=1
TRANS=counts/counts.sum(axis=1,keepdims=True)
def baseline_loss(tracks):
    return np.mean([-np.log(TRANS[TOK[a],TOK[b]]+1e-10) for tr in tracks for a,b in zip(tr[:-1],tr[1:])])
base_val=baseline_loss(val_tracks)
print(f'Baseline validation next-event cross-entropy: {base_val:.4f}')
print('Top three next events after',VOCAB[1],':',[(VOCAB[i],round(float(TRANS[TOK[VOCAB[1]],i]),3)) for i in np.argsort(TRANS[TOK[VOCAB[1]]])[::-1][:3]])

## Part 4 - LSTM: musical memory (50–60 minutes)
A first-order baseline predicts the next event using only the current event. The LSTM processes a **window of previous events**. Tokens become learned embedding vectors. The recurrent state carries information, while learned forget/input/output gates regulate its cell and hidden states. Its output layer predicts **probabilities for the next event**. During training, the true next event is the target; cross-entropy loss measures prediction error. An LSTM does **not** remember every note forever.

In symbols: `input sequence → embedding → LSTM → output logits → softmax probabilities → sampled next event`. When generating, the sampled event is appended and the process repeats.

**Check your understanding:** Why might a longer context help? Why might a 1-step transition model still be competitive? Define an epoch and distinguish training loss from validation loss.


### From an ordinary network to an LSTM
An ordinary feed-forward network receives a fixed input and produces an output, but has no built-in state passed to the next time step. An RNN processes notes in order and carries a hidden state forward. The hidden state is a numerical summary shaped by earlier inputs, not a transcript of every preceding note.

Over longer sequences, ordinary RNNs can struggle to propagate learning signals from distant steps. The LSTM adds a *cell state* plus learned gates that regulate information flow. The **forget gate** scales information carried forward; the **input gate** controls proposed new cell content; the **output gate** controls which current content contributes to the hidden state. The gates are trained from data, not handwritten musical rules.

This is like a listener noticing that a phrase often returns after several beats, but it is only an analogy: an LSTM does not literally understand musical meaning. It adjusts weights to improve next-event probabilities.

The notebook passes token embeddings through the LSTM and uses the last hidden-state output to predict the next event. When you inspect `NoteLSTM`, focus on the path **token IDs → embedding → LSTM → output scores**, not memorising PyTorch syntax.


In [ ]:
# Teaching figures: the computational data flow and a CONCEPTUAL memory-gate model.
# These schematics are not traces of actual gate activations from the trained LSTM.
from matplotlib.patches import FancyBboxPatch,FancyArrowPatch
fig,axs=plt.subplots(2,1,figsize=(12,6.2))
ax=axs[0];ax.set_xlim(0,13);ax.set_ylim(0,3);ax.axis('off')
labels=[('Earlier notes\n[60,64,67...]',.2),('Token IDs\n[3,5,6...]',2.9),('Embedding\nlearned vectors',5.5),('LSTM\nsequential memory',8.1),('Scores →\nnext note',10.7)]
for t,x in labels:
    ax.add_patch(FancyBboxPatch((x,1.05),2,1,boxstyle='round,pad=.12',facecolor='#e1ebf4',edgecolor='#37749b',linewidth=1.5))
    ax.text(x+1,1.55,t,ha='center',va='center',fontsize=10)
for x in [2.2,4.9,7.5,10.1]:ax.annotate('',xy=(x+.55,1.55),xytext=(x,1.55),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.set_title('Figure A: Data flow during next-event prediction',fontsize=13,weight='bold')
ax=axs[1];ax.axis('off');ax.set_xlim(0,13);ax.set_ylim(0,3.2)
ax.add_patch(FancyBboxPatch((4.2,1.28),4.7,.85,boxstyle='round,pad=.08',facecolor='#dff3e9',edgecolor='#278b72',linewidth=1.5))
ax.text(6.55,1.72,'Cell state: controlled information flow',ha='center',va='center',fontsize=11)
for x,title,desc in [(1.2,'Forget','scale old state'),(5.3,'Input','write new content'),(9.4,'Output','expose hidden state')]:
    ax.add_patch(FancyBboxPatch((x,.15),2.25,.7,boxstyle='round,pad=.06',facecolor='#fff1d8',edgecolor='#c28a32'))
    ax.text(x+1.125,.57,title+' gate',ha='center',fontsize=10,weight='bold')
    ax.text(x+1.125,.34,desc,ha='center',fontsize=8)
    ax.annotate('',xy=(6.2,1.24),xytext=(x+1.125,.85),arrowprops=dict(arrowstyle='->',color='#8e7857'))
ax.text(6.5,2.55,'Gate weights are learned; these are explanatory roles, not hard-coded music rules.',ha='center',fontsize=10)
ax.set_title('Figure B: Simplified LSTM memory controls',fontsize=13,weight='bold')
plt.tight_layout();plt.savefig(OUT/'lstm_teaching_diagrams.png',dpi=170);plt.show()


In [ ]:
# The dataset is fixed before experimentation; no test labels used in model selection.
CONTEXT=12

def windows(tracks,context):
    xs=[];ys=[]
    for tr in tracks:
        ids=[TOK[v] for v in tr]
        for k in range(context,len(ids)):
            xs.append(ids[k-context:k]); ys.append(ids[k])
    return torch.tensor(xs,dtype=torch.long),torch.tensor(ys,dtype=torch.long)
Xtr,ytr=windows(train_tracks,CONTEXT)
Xva,yva=windows(val_tracks,CONTEXT)
print('X training:',tuple(Xtr.shape),'y:',tuple(ytr.shape),'X validation:',tuple(Xva.shape))
print('Example events:',[REVTOK[int(t)] for t in Xtr[0]],'→ next:',REVTOK[int(ytr[0])])

class NoteLSTM(nn.Module):
    def __init__(self,n_vocab,hidden=32,embed=12):
        super().__init__(); self.emb=nn.Embedding(n_vocab,embed)
        self.lstm=nn.LSTM(embed,hidden,batch_first=True)
        self.out=nn.Linear(hidden,n_vocab)
    def forward(self,x):
        v=self.emb(x);h,_=self.lstm(v);return self.out(h[:,-1,:])

def train_model(hidden=32,epochs=6,lr=.014,seed=SEED,context=12,verbose=True):
    torch.manual_seed(seed)
    xt,yt=windows(train_tracks,context); xv,yv=windows(val_tracks,context)
    model=NoteLSTM(len(VOCAB),hidden)
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    criterion=nn.CrossEntropyLoss(); hist=[]
    for ep in range(epochs):
        model.train();perm=torch.randperm(len(xt));losses=[]
        for ids in perm.split(256):
            opt.zero_grad();l=criterion(model(xt[ids]),yt[ids]);l.backward();opt.step();losses.append(float(l.detach()))
        model.eval()
        with torch.no_grad(): vloss=float(criterion(model(xv),yv)); pred=model(xv).argmax(1);acc=float((pred==yv).float().mean())
        hist.append((ep+1,float(np.mean(losses)),vloss,acc))
        if verbose: print(f'epoch {ep+1:02d}: train_loss={hist[-1][1]:.3f}, validation_loss={vloss:.3f}, val_top1={acc:.1%}')
    return model,pd.DataFrame(hist,columns=['epoch','train_loss','val_loss','val_accuracy'])
print('Network ready. Now train in Part 5.')

## Part 5. Train your AI composer
Run this cell once for the initial LSTM. **Loss is the next-event prediction error**, not a direct rating of musical beauty. Compare LSTM validation loss with the transition baseline and note whether it is better or worse.

### How a neural network learns (read before running)
For each training example, the input is a window of earlier musical events and the target is the real next event. The model predicts scores for all possible events. **Cross-entropy loss** compares those scores with the correct target. Backpropagation calculates gradients, and the Adam optimiser adjusts learned weights using them.

One **epoch** means the training procedure has seen all its training windows once. Training loss shows fit to training examples. **Validation loss** tests predictions on different tracks excluded from fitting. A smaller training loss need not yield lower validation loss: a model may overfit repeated motifs.

When the graph appears, describe its horizontal axis (epochs) and vertical axis (loss). Examine both curves, not just the last training number. You do not need to derive gradients or reproduce the training-loop code from memory.


In [ ]:
start_time=time.time()
model,history=train_model(hidden=32,epochs=6,context=CONTEXT)
print(f'Training time: {time.time()-start_time:.1f} s | baseline val loss={base_val:.3f} | LSTM val loss={history.val_loss.iloc[-1]:.3f}')
fig,ax=plt.subplots(figsize=(8,3));ax.plot(history.epoch,history.train_loss,'-o',label='training');ax.plot(history.epoch,history.val_loss,'-o',label='validation');ax.set(xlabel='Epoch',ylabel='Cross-entropy loss',title=f'{VARIANT_ID}: LSTM learning curves');ax.legend();plt.tight_layout();plt.savefig(OUT/'training_curve.png',dpi=160);plt.show()

## Part 6 - Play raw AI events (20 minutes)
The model produces a probability distribution at each next-event step. **Temperature** changes how concentrated that distribution is; it does not improve the learned model. Listen to low, medium and high settings. These outputs are raw events with simple sound synthesis. Later we arrange a longer version of the melody into a song.


### From next-note predictions to a complete composition
The LSTM does not create a whole song in one decision. It predicts a distribution for the *next* event. We sample an event, append it to the context, then predict again. This is **autoregressive generation**, similar in spirit to next-token language generation.

**Temperature** changes the sharpness of the predicted distribution. At low values, the highest-scoring events dominate; at higher values, less likely events become more frequent. Low temperature can repeat familiar phrases; high temperature can become surprising or incoherent. Neither is always 'best'.

To compare fairly, keep the trained weights and starting phrase fixed while changing temperature. Listen to what actually happened. A numerical next-event score is not the same as a listener's judgement of musical quality.

**How to listen:** Run the generation cell. You should see a separate audio player after each track filename. Click **▶** to listen to each result. The plots appear after the audio players.


In [ ]:
def predict_next_prob(model, events, temperature=1.0):
    if temperature<=0:raise ValueError('Temperature must be > 0')
    model.eval();context=model_context.get(id(model),CONTEXT)
    if len(events)<context: events=([0]*(context-len(events)))+list(events)
    x=torch.tensor([[TOK[e] for e in events[-context:]]],dtype=torch.long)
    with torch.no_grad(): logits=model(x)[0].numpy()
    s=(logits-logits.max())/temperature; p=np.exp(s);return p/p.sum()
def generate(model,primer,n=48,temperature=1,seed=123):
    rng=np.random.default_rng(seed);seq=list(primer)
    for _ in range(n):
        p=predict_next_prob(model,seq,temperature)
        seq.append(REVTOK[int(rng.choice(len(VOCAB),p=p))])
    return seq
model_context={id(model):CONTEXT}
PRIMER=patterns[VARIANT_IN_FAMILY-1]+patterns[(VARIANT_IN_FAMILY+2)%8]
print('Provided opening motif:',PRIMER)
print('Next-event distribution after this opening:',[(VOCAB[i],round(float(p),3)) for i,p in enumerate(predict_next_prob(model,PRIMER))])
generated_tracks={}
for tag,temp in [('A_focused',.55),('B_balanced',1.0),('C_exploratory',1.45)]:
    track=generate(model,PRIMER,48,temp,SEED+int(temp*100))
    generated_tracks[tag]=track
    path=save_audio(track,f'{VARIANT_ID}_{tag}.wav')
    print(tag,'temp:',temp,'unique generated events:',len(set(track[len(PRIMER):])),'file:',path)
    display(Audio(filename=str(path)))
plot_roll(generated_tracks['B_balanced'], 'Generated composition - balanced sampling')

### Your first listening comparison
Listen to each track (headphones recommended). Are they coherent? Repetitive? Surprising? Avoid claiming that temperature alone *guarantees* originality. Compare at least two outputs and give a concrete example of what you heard.

## Part 7. Three experiments, one controlled comparison
Every group must perform these three investigations:
1. **Sampling temperature** (already performed above): change temperature without retraining the model; listen and record differences.
2. **Training duration**: compare 3 versus 8 epochs with fixed architecture and validation data.
3. **Your assigned investigation**: for your variant, compare either hidden-state size (16 vs 48) or sequence context (8 vs 20).

We fix random seeds within model comparisons so results are reproducible. **Model-selection scores are validation metrics** and should not be described as independent final test scores. Interpret failed experiments as carefully as successful ones.

In [ ]:
# Experiment 2: identical model hyperparameters except epochs
short_model, short_hist = train_model(hidden=32,epochs=3,seed=SEED,verbose=False)
long_model, long_hist = train_model(hidden=32,epochs=8,seed=SEED,verbose=False)
# Experiment 3: variant-specific parameter comparison
if VARIANT % 2 == 0:
    ASSIGNED_FACTOR='hidden units'
    exp_a,ha=train_model(hidden=16,epochs=5,seed=SEED,verbose=False)
    exp_b,hb=train_model(hidden=48,epochs=5,seed=SEED,verbose=False)
    value_a,value_b=16,48
else:
    ASSIGNED_FACTOR='context length'
    exp_a,ha=train_model(hidden=32,epochs=5,context=8,seed=SEED,verbose=False)
    exp_b,hb=train_model(hidden=32,epochs=5,context=20,seed=SEED,verbose=False)
    value_a,value_b=8,20
comparison=pd.DataFrame([
 ['baseline transition',None,base_val,None],
 ['initial LSTM (6 epochs)',6,float(history.val_loss.iloc[-1]),float(history.val_accuracy.iloc[-1])],
 ['epochs short',3,float(short_hist.val_loss.iloc[-1]),float(short_hist.val_accuracy.iloc[-1])],
 ['epochs long',8,float(long_hist.val_loss.iloc[-1]),float(long_hist.val_accuracy.iloc[-1])],
 [f'{ASSIGNED_FACTOR}={value_a}',5,float(ha.val_loss.iloc[-1]),float(ha.val_accuracy.iloc[-1])],
 [f'{ASSIGNED_FACTOR}={value_b}',5,float(hb.val_loss.iloc[-1]),float(hb.val_accuracy.iloc[-1])],
],columns=['condition','epochs','validation_loss','validation_accuracy'])
print('Assigned experiment:',ASSIGNED_FACTOR,value_a,'versus',value_b)
display(comparison)
comparison.to_csv(OUT/'experiment_comparison.csv',index=False)
fig,ax=plt.subplots(figsize=(9,3.5));temp_comp=comparison.iloc[1:];ax.barh(temp_comp.condition,temp_comp.validation_loss);ax.set(xlabel='Validation loss (lower better)',title=f'{VARIANT_ID} Controlled experiments');plt.tight_layout();plt.savefig(OUT/'experiment_graph.png',dpi=160);plt.show()

**Report experiment task (20 marks):** Present the metrics table, graph, and at least three observations with numeric evidence. Which run would you choose for next-note prediction and why? What can the metrics *not* tell you about the music? For temperature, compare audible outputs and diversity (you may calculate number of unique tokens). Note the training run comparisons use separately trained models and the temperature experiment reuses one fixed model.

## Part 8 - Create your own musical opening (60–90 minutes)

### What are you making?

A **motif** is a short musical idea: a few notes or drum hits that form a pattern. Think of the opening of a familiar tune. Even if the whole song is long, the first few notes might be recognisable. **Your task is to create a short opening, listen to it, and then ask the trained LSTM to continue it.**

We represent music using **events**. In this notebook, one event is one step on a musical timeline. It can be a note, a drum sound, or silence (a **rest**). An **8-event motif** is simply eight choices in order. You do not need to know music theory or MIDI numbers.

### What you actually need to do

1. Run the next cell once. It will print a **menu of available musical sounds** for your group's assigned music style. For percussion groups these are drum sounds; for other groups they are notes.
2. Look for the line `MY_PATTERN = [...]`. It contains eight numbers. **Only edit the numbers inside those square brackets.** A number from **1 up to the highest number shown** chooses a sound from the displayed menu; **0 means silence/rest**.
3. Keep **8 to 16 numbers** separated by commas. Example: `[1, 2, 3, 0, 3, 2, 1, 0]` plays three sounds, pauses, then plays three more sounds and pauses.
4. Run the cell again. You will see your pattern on a graph and an **audio player**. Click its **▶ Play** button to hear your opening. If you dislike it, change two or three numbers and run again.
5. When happy, continue to the *next* cell. That next cell lets the LSTM **continue your opening** with new events. You will hear your opening and the AI continuation together.

**Important:** You do **not** need to edit `VOCAB`, `MY_MOTIF`, or the sound-generation functions. The notebook converts your simple choices into the numerical representation needed by the model.

**Answer in your report:** Describe your opening in everyday language. Which sounds repeat? Where did you place a rest? What changed between your first attempt and final opening? What did the LSTM add that you did not enter yourself?

### Optional: Use a MIDI melody instead

If your group has a short `.mid` or `.midi` melody, the same cell offers an optional upload switch. Leave `USE_MIDI_UPLOAD = False` unless you specifically want this. A MIDI file contains note instructions, unlike an MP3 recording. It is fine (and easier) to create your own eight-number motif. **MP3/WAV uploads belong in the later Bring Your Own Instrumental section**, not in this MIDI step.

In [ ]:
# YOUR GROUP'S MUSICAL OPENING - BEGINNER EDITING CELL
# 1. Run this cell once to see the numbered choices below.
# 2. Change ONLY the numbers in MY_PATTERN.
# 3. Run again, then CLICK PLAY under the graph.

# 0 = rest (silence); other valid numbers are listed below.
MY_PATTERN = [1, 2, 3, 0, 4, 3, 2, 1]  # use 8 to 16 numbers, chosen from the displayed menu

# The menu is tailored to YOUR group's musical family.
def simple_note_label(midi_pitch):
    names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    return names[int(midi_pitch) % 12] + str(int(midi_pitch) // 12 - 1)

if FAMILY == 'Percussion grooves':
    drum_names = {1: 'Kick (low drum)', 2: 'Snare (sharp drum)',
                  3: 'Hi-hat (short tick)', 4: 'Extra percussion hit'}
    sound_names = {i: drum_names.get(int(p), 'Drum') for i,p in enumerate(VOCAB) if i > 0}
else:
    sound_names = {i: simple_note_label(p) for i,p in enumerate(VOCAB) if i > 0}

print('YOUR SOUND MENU (these numbers are choices, not MIDI notes):')
print('0 = Rest / silence')
for number, sound in sound_names.items():
    print(f'{number} = {sound}')

if not isinstance(MY_PATTERN, (list, tuple)) or not 8 <= len(MY_PATTERN) <= 16:
    raise ValueError('MY_PATTERN needs 8 to 16 numbers, for example [1,2,3,0,4,3,2,1].')
if any(not isinstance(x, int) or x < 0 or x >= len(VOCAB) for x in MY_PATTERN):
    raise ValueError('Use only whole numbers shown in the menu: 0 through '+str(len(VOCAB)-1))

# Automatic conversion into actual musical events. No edits needed below.
MY_MOTIF = [VOCAB[choice] for choice in MY_PATTERN]
print('Your chosen sound numbers:', MY_PATTERN)
print('Your pattern as music events:', MY_MOTIF)
plot_roll(MY_MOTIF, 'Your group-created musical opening')
print('CLICK THE PLAY BUTTON below to hear your opening:')
display(Audio(data=render_events(MY_MOTIF)[0], rate=16000))

# OPTIONAL MIDI FILE: Change this to True only if you have a .mid/.midi file.
# This REPLACES your numbered opening with the melody extracted from the file.
USE_MIDI_UPLOAD = False
if USE_MIDI_UPLOAD:
    from google.colab import files
    print('Select a MIDI file (.mid or .midi), NOT an MP3/WAV audio recording.')
    uploaded = files.upload()
    midi_path = next((f for f in uploaded if f.lower().endswith(('.mid','.midi'))), None)
    if midi_path is None:
        raise ValueError('Please select a .mid or .midi file, or switch USE_MIDI_UPLOAD back to False.')
    pm = pretty_midi.PrettyMIDI(midi_path)
    notes = sorted([note for inst in pm.instruments if not inst.is_drum for note in inst.notes],
                   key=lambda note: note.start)
    if not notes:
        raise ValueError('No pitched melody notes were found in this MIDI file.')
    chosen = {}
    for note in notes:
        step = max(0, round(note.start / .24))
        chosen.setdefault(step, min(VOCAB[1:], key=lambda p: abs(p-note.pitch)))
    first = min(chosen)
    MY_MOTIF = [chosen.get(k,0) for k in range(first, first+min(16,max(8,max(chosen)-first+1)))]
    print('Extracted your MIDI opening:', MY_MOTIF)
    plot_roll(MY_MOTIF,'Opening extracted from MIDI')
    print('CLICK PLAY to hear the extracted opening:')
    display(Audio(data=render_events(MY_MOTIF)[0],rate=16000))

In [ ]:
# CONTINUE YOUR OWN MUSIC, using the trained LSTM
personal=generate(model,MY_MOTIF,n=48,temperature=.9,seed=SEED+999)
personal_path=save_audio(personal,f'{VARIANT_ID}_personal_continuation.wav')
print('Your motif:',MY_MOTIF, 'AI continuation:',personal[len(MY_MOTIF):])
plot_roll(personal,'Personal motif + AI continuation')
display(Audio(filename=str(personal_path)))

def export_midi(events,filename,family=FAMILY,step_sec=.24):
    pm=pretty_midi.PrettyMIDI(initial_tempo=125)
    inst=pretty_midi.Instrument(program=0,is_drum=(family=='Percussion grooves'))
    drum_map={1:36,2:38,3:42,4:39}
    for j,e in enumerate(events):
        if e==0:continue
        pitch=drum_map.get(e,36) if family=='Percussion grooves' else int(e)
        inst.notes.append(pretty_midi.Note(velocity=88,pitch=pitch,start=j*step_sec,end=(j+.85)*step_sec))
    pm.instruments.append(inst);pm.write(str(OUT/filename));return OUT/filename
for tag,track in generated_tracks.items():export_midi(track,f'{VARIANT_ID}_{tag}.mid')
export_midi(personal,f'{VARIANT_ID}_personal_continuation.mid')
print('WAV and MIDI files are saved in',OUT)
print('Required: select your best two sampled tracks and your personal continuation for presentation.')

## Part 9 - From AI notes to a finished track (90 minutes)
**The creative studio.** Your trained LSTM supplies the **lead sequence**. The code below adds a simple music arrangement: repeated motifs, bass notes, triad accompaniment, drums, an intro, main section, variation and ending. It renders a **longer WAV** using lightweight synthesis, and writes separate MIDI tracks.

**Students can change:** BPM, chosen instrument sound, chord/bass volumes, arrangement intensity, and number of bars. Try at least **two arrangement changes** and report what you heard. This step is *algorithmic arrangement*, not proof of stronger LSTM learning.

**Required evidence:** play `raw_AI_only.wav` and `final_AI_arrangement.wav` in your presentation, and explain which parts were generated by the LSTM and which were added by rules.


### Where does a 'finished' song come from?
The first LSTM output is only a sequence of lead events. Arrangement code can repeat musical phrases, introduce sections, put bass underneath, add chords and drum beats, and synthesise an audio waveform. These additions may make the result more pleasant, but they are *not extra neural-network predictions*.

**You must compare two exports:** the raw LSTM events and the arranged studio track. Describe which audible changes came from the learned model and which from deterministic rules. In a presentation, the comparison is more educational than playing only the finished arrangement.


In [ ]:
# STUDIO 2.0: longer arrangements; no external music files or SoundFont downloads.
from scipy.signal import fftconvolve
STUDIO_BPM = 108    # STUDENTS: try 90, 108, 125 (beat speed)
BASS_VOLUME = 0.22  # STUDENTS: try values from .08 to .32
CHORD_VOLUME = 0.14 # STUDENTS: try values from .05 to .22
DRUM_VOLUME = 0.20  # STUDENTS: try values from .08 to .28
STEPS_PER_BEAT = 2
SECONDS_PER_STEP = 60.0 / STUDIO_BPM / STEPS_PER_BEAT
RATE = 16000

def choose_phrase(tr, n=32):
    # Use LSTM-generated material only, not the prefix; deterministically wrap if needed.
    ai=tr[len(PRIMER):]
    return (ai * (1+n//max(1,len(ai))))[:n]

def arrange_structure(track):
    phrase=choose_phrase(track,32)
    p2=phrase[8:]+phrase[:8]
    # 8 x 16-event sections; 128 events (~36 seconds at 108 BPM).
    # 10 x 16-event sections -> ~44.4 seconds at 108 BPM.
    whole=(phrase[:16]+phrase[:16]+phrase[16:]+p2[:16]+phrase[:16]+p2[16:]+phrase[:16]+phrase[16:]+p2[:16])[:160]
    return whole

def synth_tone(pitch, dur, instrument='piano', sr=RATE):
    n=max(1,int(sr*dur)); t=np.arange(n)/sr
    f=440.0*(2**((pitch-69)/12))
    if instrument=='piano':
        wave=(np.sin(2*np.pi*f*t)+.35*np.sin(2*np.pi*2*f*t)+.10*np.sin(2*np.pi*3*f*t))
        env=(1-np.exp(-90*t))*np.exp(-3.1*t)
    elif instrument=='marimba':
        wave=(np.sin(2*np.pi*f*t)+.30*np.sin(2*np.pi*3.98*f*t)+.16*np.sin(2*np.pi*9.2*f*t))
        env=(1-np.exp(-110*t))*np.exp(-8*t)
    elif instrument=='synth':
        wave=.68*np.sin(2*np.pi*f*t)+.20*np.sin(2*np.pi*2.01*f*t)+.15*np.sin(2*np.pi*.5*f*t)
        env=np.minimum(1,t/.045)*np.minimum(1,(dur-t)/.12)
    else:
        wave=np.sin(2*np.pi*f*t)+.18*np.sin(2*np.pi*2*f*t)
        env=np.minimum(1,t/.025)*np.exp(-1.5*t)
    return (wave*np.maximum(env,0)).astype(np.float32)

def drum_sound(which, n, sr=RATE, seed=0):
    r=np.random.default_rng(seed);t=np.arange(n)/sr
    if which=='kick': return (np.sin(2*np.pi*(75*t-25*t*t))*np.exp(-23*t)).astype(np.float32)
    if which=='snare': return (.7*r.normal(size=n)*np.exp(-24*t)+.2*np.sin(2*np.pi*190*t)*np.exp(-18*t)).astype(np.float32)
    return (r.normal(size=n)*np.exp(-75*t)).astype(np.float32)

def synth_arrangement(melody, filename, family=FAMILY, bpm=STUDIO_BPM, bass_volume=BASS_VOLUME, chord_volume=CHORD_VOLUME, drum_volume=DRUM_VOLUME):
    step=60/bpm/2; sr=RATE; length=int((len(melody)*step+1.2)*sr)
    stems={k:np.zeros(length,dtype=np.float32) for k in ['lead','bass','chords','drums']}
    def add(stem,wave,t0,volume):
        start=int(t0*sr);end=min(length,start+len(wave))
        if end>start:stems[stem][start:end]+=volume*wave[:end-start]
    pitches=[x for x in melody if x in range(12,128)]
    root_note=60 if family=='Percussion grooves' else (min(pitches,key=lambda x:abs(x-60)) if pitches else 60)
    scale=PITCH_CLASSES.get(family,[60,62,64,65,67,69,71,72])
    for i,event in enumerate(melody):
        tm=i*step; progress=i/max(1,len(melody)); fade=min(1,progress*9,(1-progress)*13)
        lead_volume=.28*max(.03,fade)
        if family=='Percussion grooves':
            # LSTM event is a rhythmic percussion token; note 1=kick, 2=snare, 3=hat, 4=clap
            drum_type={1:'kick',2:'snare',3:'hat',4:'snare'}.get(event)
            if drum_type: add('lead',drum_sound(drum_type,int(.21*sr),seed=SEED+i),tm,lead_volume)
        elif event!=0:
            tone='synth' if family=='Electronic motifs' else 'marimba' if family=='Folk-style melodies' else 'piano'
            add('lead',synth_tone(int(event),step*.95,tone),tm,lead_volume)
        # 16-step bar changes chord. Triads and bass are RULE-BASED, not generated by the LSTM.
        if i%8==0:
            chordroot=(root_note+[0,5,7,3][(i//16)%4])
            chordroot=max(48,min(71,chordroot))
            add('bass',synth_tone(chordroot-12,step*6,'synth'),tm,bass_volume*fade)
            for delta in (0,4,7):add('chords',synth_tone(chordroot+delta,step*7,'soft'),tm,chord_volume*fade/3)
        if i%4==0:add('drums',drum_sound('kick',int(.2*sr),seed=SEED+i+999),tm,drum_volume*fade)
        if i%8==4:add('drums',drum_sound('snare',int(.18*sr),seed=SEED+i+123),tm,drum_volume*.75*fade)
        if i%2==0:add('drums',drum_sound('hat',int(.1*sr),seed=SEED+i+234),tm,drum_volume*.22*fade)
    # Tiny room echo for a more finished sound without downloads.
    mix=sum(stems.values())
    delay=int(.17*sr)
    if delay<len(mix):mix[delay:]+=.11*mix[:-delay].copy()
    scale=max(.95,np.max(np.abs(mix))/.94)
    wavfile.write(str(OUT/filename),sr,np.int16(np.clip(mix/scale,-1,1)*32767))
    return OUT/filename,stems,step

def midi_arrangement(melody,filename,family=FAMILY,bpm=STUDIO_BPM):
    step=60/bpm/2
    pm=pretty_midi.PrettyMIDI(initial_tempo=bpm)
    lead=pretty_midi.Instrument(80 if family=='Electronic motifs' else 12 if family=='Folk-style melodies' else 0,is_drum=(family=='Percussion grooves'),name='AI-generated lead')
    bass=pretty_midi.Instrument(33,name='Rule-based bass')
    chords=pretty_midi.Instrument(0,name='Rule-based chords')
    drums=pretty_midi.Instrument(0,is_drum=True,name='Rule-based drums')
    pitches=[p for p in melody if p>=12];root=60 if family=='Percussion grooves' else min(pitches,key=lambda x:abs(x-60)) if pitches else 60
    for i,event in enumerate(melody):
        tm=i*step
        if event!=0:
            pitch={1:36,2:38,3:42,4:39}.get(event,36) if family=='Percussion grooves' else int(event)
            lead.notes.append(pretty_midi.Note(90,pitch,tm,tm+step*.85))
        if i%8==0:
            r=max(48,min(71,root+[0,5,7,3][(i//16)%4]))
            bass.notes.append(pretty_midi.Note(70,r-12,tm,tm+step*6))
            for d in (0,4,7):chords.notes.append(pretty_midi.Note(55,r+d,tm,tm+step*7))
        if i%4==0:drums.notes.append(pretty_midi.Note(74,36,tm,tm+.12))
        if i%8==4:drums.notes.append(pretty_midi.Note(65,38,tm,tm+.12))
        if i%2==0:drums.notes.append(pretty_midi.Note(48,42,tm,tm+.08))
    for inst in [lead,bass,chords,drums]: pm.instruments.append(inst)
    pm.write(str(OUT/filename));return OUT/filename

# Repeatable sample: SAME LSTM, three temperature choices, and a student-created motif.
# Extend learned material to a long song; accompaniment is rule-based.
ARRANGED={}
for tag in ['A_focused','B_balanced','C_exploratory']:
    seq=arrange_structure(generated_tracks[tag])
    wav,stems,step=synth_arrangement(seq,f'{VARIANT_ID}_{tag}_STUDIO.wav')
    midi_arrangement(seq,f'{VARIANT_ID}_{tag}_STUDIO.mid')
    ARRANGED[tag]=seq
    print(tag,'duration',round(len(seq)*step,1),'seconds; file:',wav)
    display(Audio(filename=str(wav)))
# Personal motif continuation uses same trained LSTM; extend its predicted material.
personal_primer_length=len(MY_MOTIF)
custom_ai=personal[personal_primer_length:]
custom_long=(custom_ai*4)[:160]
custom_wav,_,_=synth_arrangement(custom_long,f'{VARIANT_ID}_personal_STUDIO.wav')
midi_arrangement(custom_long,f'{VARIANT_ID}_personal_STUDIO.mid')
raw_path=save_audio(personal,f'{VARIANT_ID}_raw_AI_only.wav')
print('RAW LSTM:',raw_path,'PERSONAL FINISHED:',custom_wav)
display(Audio(filename=str(raw_path)))
display(Audio(filename=str(custom_wav)))


## Part 10 - Investigate production choices (30–45 minutes)
Change at least two settings in the studio cell (for example `STUDIO_BPM`, `BASS_VOLUME`, `CHORD_VOLUME`, `DRUM_VOLUME`), rerun that cell and export an alternate mix. Explain audible differences. Keep the original mix as evidence.

**Critical investigation:** A good arrangement can hide weak melodic predictions. Compare raw versus arranged versions and explain whether improvement came from the LSTM or from sound design and regular musical backing. Musical quality is subjective: use listening ratings from at least three group members and give one concrete observation per listener.

**Report:** (1) identify learned vs rule-based elements; (2) include training/validation curves, two controlled comparisons, plus temperature study; (3) include one personal motif with evidence; (4) embed links to finished WAV/MIDI files; (5) describe two limitations, such as repetitive motifs and difficult quantisation of uploaded MIDI.


---
# Part 10A - Bring Your Own Instrumental: From waveform to beats and key

**Learning objective:** understand how an ordinary recording differs from MIDI; inspect it visually; estimate beat timing and tonal context; recognise uncertainty. This section **adds to** all the previous LSTM and composition sections.

### What you will do
1. Upload a short instrumental recording in MP3, WAV or FLAC, **or use the built-in royalty-free synthetic backing** generated in the next cell. Keep music to 30–60 seconds and under 25 MB.
2. Listen to it and plot its waveform. A waveform is amplitude over time; tall peaks are not automatically 'beats'.
3. Estimate tempo in beats per minute (BPM), locate likely beats and inspect a chromagram. A chromagram groups pitched audio energy into twelve pitch classes (C through B), ignoring octave.
4. Estimate a *possible* musical key. This is a rough heuristic, not guaranteed key recognition: mixtures, drums and chord changes can make the estimate unreliable. The notebook therefore lets you choose BPM and key manually.

**Beginner question:** Why can we not treat every MP3 audio sample as if it were a MIDI note token? Why might percussion confuse musical-key estimation?

**Copyright:** Use recordings you made or have permission to remix and share, or use the supplied backing. Do not redistribute copyrighted recordings without permission.

**Your own instrumental:** Later you can set `UPLOAD_AUDIO = True` to select an MP3/WAV/FLAC file in Colab. The notebook then shows a player for the uploaded track. The default is a built-in example, so everyone can complete the task even without a music file.


In [ ]:
# AUDIO INPUT - run without editing to use the built-in sample.
# In Colab, set UPLOAD_AUDIO=True to choose an MP3, WAV or FLAC from your computer.
UPLOAD_AUDIO = False
LOCAL_AUDIO_PATH = ''  # Optional: enter an existing local path in desktop Jupyter
MAX_AUDIO_SECONDS = 52
MAX_UPLOAD_MB = 25

import soundfile as sf
from scipy import signal

def create_fallback_backing():
    # A deterministic backing made from studio accompaniment WITHOUT the LSTM lead.
    melody = ARRANGED['B_balanced']
    _, stems, step = synth_arrangement(melody,f'{VARIANT_ID}_temporary_fullmix.wav')
    backing = stems['bass'] + stems['chords'] + stems['drums']
    # A gentle echo for a more realistic instrumental.
    echo=int(.145*RATE)
    if echo<len(backing): backing[echo:] += .095*backing[:-echo].copy()
    backing = np.clip(backing/max(1,np.max(np.abs(backing))/.82),-.95,.95)
    path=OUT/f'{VARIANT_ID}_BUILTIN_INSTRUMENTAL.wav'
    sf.write(path,backing,RATE)
    return path

if UPLOAD_AUDIO:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError('Colab upload is available in Google Colab. For desktop Jupyter set LOCAL_AUDIO_PATH instead.')
    uploaded=files.upload()
    paths=[pathlib.Path(x) for x in uploaded if pathlib.Path(x).suffix.lower() in ('.mp3','.wav','.flac')]
    if not paths: raise ValueError('Please choose an MP3, WAV or FLAC file.')
    source_audio=paths[0]
    if source_audio.stat().st_size>MAX_UPLOAD_MB*1024*1024:raise ValueError('Audio exceeds 25 MB; shorten it first.')
elif LOCAL_AUDIO_PATH.strip():
    source_audio=pathlib.Path(LOCAL_AUDIO_PATH)
    if not source_audio.exists():raise FileNotFoundError(source_audio)
else:
    source_audio=create_fallback_backing()

# librosa can decode MP3 in modern Colab; for unsupported codecs, convert to WAV externally.
import librosa
BACKING_SR=16000
backing_audio, BACKING_SR = librosa.load(str(source_audio),sr=BACKING_SR,mono=True,duration=MAX_AUDIO_SECONDS)
if len(backing_audio)<BACKING_SR*4:raise ValueError('Audio must be at least four seconds long.')
BACKING_DURATION=len(backing_audio)/BACKING_SR
print('Loaded:',source_audio.name,'duration:',round(BACKING_DURATION,1),'s; samples:',len(backing_audio),'sample rate:',BACKING_SR)
output_original=OUT/f'{VARIANT_ID}_UPLOAD_ORIGINAL_EXCERPT.wav'
sf.write(output_original,backing_audio,BACKING_SR)
display(Audio(data=backing_audio,rate=BACKING_SR))
fig,ax=plt.subplots(figsize=(11,2.5)); tt=np.arange(len(backing_audio))/BACKING_SR
ax.plot(tt[::64],backing_audio[::64],linewidth=.6);ax.set(xlabel='Seconds',ylabel='Amplitude',title='Your instrumental: waveform');plt.tight_layout();plt.savefig(OUT/'uploaded_waveform.png',dpi=150);plt.show()


## Part 10B - Finding beats and estimating tonal context
**Beat tracking** looks for rhythmic regularity, often using changes in sound energy. The estimated tempo is measured in beats per minute (BPM). For example, 120 BPM implies roughly half a second per beat. This is a useful approximation for aligning notes, but real recordings can contain tempo changes and offbeat events.

**Pitch class / chromagram:** C3, C4 and C5 are different octaves of C, so they contribute to the same chroma bin. A simple key estimate compares chroma to rough major/minor pitch-class templates. This is only a learning heuristic, not a music-theory oracle. A percussion-heavy recording may have no clearly defined key.

**Checkpoint:** Record estimated BPM, guessed key, and one reason each estimate could be wrong. Do not simply report numbers without explaining their uncertainty.


In [ ]:
# Quick, explainable signal analysis using short-time Fourier transforms and energy peaks.
# This avoids the heavy optional high-resolution harmonic/percussive decomposition.
from scipy.signal import stft, find_peaks
FRAME=512
env=np.array([np.sqrt(np.mean(backing_audio[k:k+FRAME]**2)) for k in range(0,len(backing_audio)-FRAME,FRAME)])
t_env=np.arange(len(env))*FRAME/BACKING_SR
novelty=np.maximum(0,np.diff(env,prepend=env[0]))
peaks,_=find_peaks(novelty,distance=max(1,int(.22*BACKING_SR/FRAME)),prominence=max(.001,float(np.quantile(novelty,.72))*.65))
peak_times=t_env[peaks]
if len(peak_times)>3:
    diffs=np.diff(peak_times);diffs=diffs[(diffs>.25)&(diffs<1.2)]
    bpm_est=float(60/np.median(diffs)) if len(diffs) else float(STUDIO_BPM)
else:bpm_est=float(STUDIO_BPM)
if bpm_est<65:bpm_est*=2
if bpm_est>180:bpm_est/=2
if not 40<=bpm_est<=220:bpm_est=float(STUDIO_BPM)
beat_times=np.arange(0,BACKING_DURATION,60/bpm_est)
# Short-time spectrum mapped into chroma bins; chord-rich backing is reasonably interpretable.
frequencies,frames,Z=stft(backing_audio,fs=BACKING_SR,nperseg=4096,noverlap=3072)
mag=np.abs(Z)
chromagram=np.zeros((12,len(frames)),dtype=float)
for k,f in enumerate(frequencies):
    if 100<=f<=1900:
        midi=int(round(69+12*np.log2(f/440)))
        chromagram[midi%12]+=mag[k]**1.2
chromagram/=np.maximum(1e-8,chromagram.max(axis=0,keepdims=True))
mean_chroma=chromagram.mean(axis=1)
notes12=['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
major=np.array([1,0,1,0,1,1,0,1,0,1,0,1],dtype=float)
minor=np.array([1,0,1,1,0,1,0,1,1,0,1,0],dtype=float)
candidates=[]
for root in range(12):
    for mode,pat in [('major',major),('minor',minor)]:
        template=np.roll(pat,root)
        score=float(np.dot(mean_chroma,template)/((np.linalg.norm(template)*np.linalg.norm(mean_chroma))+1e-8))
        candidates.append((score,root,mode))
KEY_SCORE,KEY_ROOT,KEY_MODE_ESTIMATED=max(candidates)
print(f'Estimated tempo: {bpm_est:.1f} BPM | Candidate beat markers: {len(beat_times)}')
print('Estimated key:',notes12[KEY_ROOT],KEY_MODE_ESTIMATED,'(heuristic similarity:',round(KEY_SCORE,3),')')
fig,axs=plt.subplots(2,1,figsize=(11,6))
axs[0].plot(tt[::128],backing_audio[::128],linewidth=.6)
for beat in beat_times[:100]:axs[0].axvline(beat,color='tab:orange',alpha=.22)
axs[0].set(xlabel='Seconds',ylabel='Amplitude',title='Estimated regular beat grid (approximation)')
axs[1].imshow(chromagram,origin='lower',aspect='auto',extent=[0,BACKING_DURATION,-.5,11.5],cmap='magma')
axs[1].set(yticks=range(12),yticklabels=notes12,xlabel='Seconds',ylabel='Pitch class',title='Approximate STFT chromagram')
plt.tight_layout();plt.savefig(OUT/'audio_beat_chroma.png',dpi=155);plt.show()


---
# Part 11 - AI Remix Laboratory: match LSTM output to the instrumental

We now combine **two different computational workflows**. The earlier LSTM predicts symbolic next events from its *training melodies*. The new audio-analysis pipeline estimates where beats occur and which scale could fit the backing. Our code places AI events on those beats, optionally remaps them into the selected scale, synthesises the resulting lead, and mixes it with the uploaded instrumental.

**This is not the LSTM learning from the uploaded MP3.** It is the LSTM composing over an instrumental using audio-derived context and ordinary arrangement rules. The distinction is essential in your report.

### Listening experiments
- Remix A: focused sampling temperature 0.55, scale-adjusted lead.
- Remix B: balanced sampling temperature 1.0, scale-adjusted lead.
- Remix C: exploratory sampling temperature 1.45, scale-adjusted lead.
- An optional *unconstrained* comparison shows how much the scale adjustment changes the notes.

Adjust **manual BPM, musical key, instrument and relative lead volume** if the defaults sound awkward. Your choices are part of the investigation. Record what changed and why.


In [ ]:
# STUDENT CONTROLS. None of these controls retrains the LSTM.
MANUAL_BPM = None        # e.g. 108 or None to use automatic tempo estimate
MANUAL_KEY = None        # e.g. 'C', 'G', 'A#', or None for automatic estimate
KEY_MODE = None          # 'major', 'minor', or None for automatic estimate
REMIX_INSTRUMENT = 'piano'  # piano, marimba, synth, soft
LEAD_VOLUME = .22       # suggested .08 to .35; keep the backing audible
ALIGN_USING_DETECTED_BEATS = True  # False uses steady BPM grid

BPM_USED=float(MANUAL_BPM if MANUAL_BPM is not None else bpm_est)
if not 40<=BPM_USED<=220:raise ValueError('BPM must be between 40 and 220')
KEY_USED=(MANUAL_KEY or notes12[KEY_ROOT]).upper()
if KEY_USED not in notes12:raise ValueError('Key must be one of '+', '.join(notes12))
ROOT_USED=notes12.index(KEY_USED)
MODE_USED=KEY_MODE if KEY_MODE in ('major','minor') else KEY_MODE_ESTIMATED
if REMIX_INSTRUMENT not in ('piano','marimba','synth','soft'):raise ValueError('Unrecognised instrument')
if not 0<=LEAD_VOLUME<=.5:raise ValueError('Use volume between 0 and 0.5')
print('Remix controls:',BPM_USED,'BPM;',KEY_USED,MODE_USED,';instrument:',REMIX_INSTRUMENT,';lead volume:',LEAD_VOLUME)


In [ ]:
# ARRANGE: pitch remapping and beat alignment are rule-based, not learned by the LSTM.
def nearest_scale_pitch(event,root=ROOT_USED,mode=MODE_USED):
    if event==0:return 0
    scale=major if mode=='major' else minor
    allowed={((root+i)%12) for i,v in enumerate(scale) if v}
    return min(range(48,85),key=lambda p:(p%12 not in allowed,abs(p-int(event)))) if int(event)>=12 else 60+root

def build_lead_from_ai(events,constrain_scale=True):
    # Select model-predicted events, not fixed intro, and repeat if needed to cover the uploaded excerpt.
    raw=list(events[len(PRIMER):])
    if not raw:raise ValueError('No generated notes available')
    desired_steps=int(np.ceil(BACKING_DURATION*BPM_USED/60*2))
    raw=(raw*((desired_steps//len(raw))+2))[:desired_steps]
    if FAMILY=='Percussion grooves':
        # Drum-token LSTM predicts RHYTHM. Map each hit to a pitched accent for remixing.
        pitch_map={0:0,1:60+ROOT_USED,2:64+ROOT_USED,3:67+ROOT_USED,4:72+ROOT_USED}
        notes=[pitch_map[int(p)] for p in raw]
    else:notes=[int(p) for p in raw]
    return [nearest_scale_pitch(p) if constrain_scale else p for p in notes]

def render_remix(tag,events,constrain_scale=True):
    notes=build_lead_from_ai(events,constrain_scale)
    lead=np.zeros(len(backing_audio),dtype=np.float32)
    step_sec=60/BPM_USED/2
    note_starts=[]
    for i,p in enumerate(notes):
        if p==0:continue
        if ALIGN_USING_DETECTED_BEATS and len(beat_times)>=4:
            beatidx=i//2; t=float(beat_times[min(beatidx,len(beat_times)-1)])
            if i%2 and beatidx+1<len(beat_times):t+=float(beat_times[beatidx+1]-beat_times[beatidx])/2
            elif i%2:t+=step_sec
        else:t=i*step_sec
        start=round(t*BACKING_SR)
        if not 0<=start<len(lead):continue
        note_starts.append((p,t))
        wave=synth_tone(p,min(step_sec*.90,.48),REMIX_INSTRUMENT,sr=BACKING_SR)
        end=min(len(lead),start+len(wave));lead[start:end]+=LEAD_VOLUME*wave[:end-start]
    # Gentle limiting avoids digital clipping; don't blindly normalise the lead to full loudness.
    mix=.88*backing_audio+lead
    peak=max(1,np.max(np.abs(mix))/.94);mix=np.clip(mix/peak,-.99,.99).astype(np.float32)
    wavpath=OUT/f'{VARIANT_ID}_{tag}_UPLOADED_REMIX.wav'
    sf.write(wavpath,mix,BACKING_SR)
    midipath=OUT/f'{VARIANT_ID}_{tag}_AI_LEAD.mid'
    pm=pretty_midi.PrettyMIDI(initial_tempo=BPM_USED)
    inst=pretty_midi.Instrument(program={'piano':0,'marimba':12,'synth':80,'soft':89}[REMIX_INSTRUMENT],name='LSTM lead aligned by program')
    for p,t in note_starts:
        inst.notes.append(pretty_midi.Note(velocity=88,pitch=min(127,max(0,int(p))),start=t,end=t+step_sec*.8))
    pm.instruments.append(inst);pm.write(str(midipath))
    return wavpath,notes

REMIX_RESULTS={}
for tag,temp in [('A_focused',.55),('B_balanced',1.0),('C_exploratory',1.45)]:
    new_seq=generate(model,PRIMER,80,temp,seed=SEED+333+int(temp*100))
    wavpath,notes=render_remix(tag,new_seq,constrain_scale=True)
    REMIX_RESULTS[tag]={'temperature':temp,'file':str(wavpath),'notes':notes,'unique_notes':len(set(x for x in notes if x))}
    print(tag,'duration',round(BACKING_DURATION,1),'seconds; distinct pitches:',REMIX_RESULTS[tag]['unique_notes'])
    display(Audio(filename=str(wavpath)))

# A genuine A/B experiment for explaining the effect of scale mapping.
raw_variant=generate(model,PRIMER,80,1.0,seed=SEED+433)
raw_comp_path,raw_comp_notes=render_remix('D_unconstrained',raw_variant,constrain_scale=False)
print('Scale constrained vs unconstrained',len(set(REMIX_RESULTS['B_balanced']['notes'])),len(set(raw_comp_notes)))
display(Audio(filename=str(raw_comp_path)))

fig,ax=plt.subplots(figsize=(12,3))
for row,(label,res) in enumerate(REMIX_RESULTS.items()):
    for i,p in enumerate(res['notes']):
        if p:ax.scatter(i,p,s=9,marker='s',alpha=.7,label=label if i==0 else None)
ax.set(xlabel='Musical event number',ylabel='MIDI pitch',title='AI-generated notes placed over the instrumental')
plt.tight_layout();plt.savefig(OUT/'remix_notes.png',dpi=160);plt.show()


## Part 11B - Listen like an AI investigator
You should now have the original instrumental, three scale-matched remixes, and an unconstrained comparison. Listen on headphones. Record a short observation for each. **Do not claim a given temperature is necessarily more creative:** describe what your group's actual track sounds like.

**Required group investigation:**
1. What did the beat detector estimate, and did the generated lead align well? If not, try a manual BPM and explain the change.
2. Listen to the scale-constrained and unconstrained versions. Can you hear a difference? Does either clash with the accompaniment?
3. Compare focused, balanced and exploratory sampling using two criteria: rhythmic coherence and note variety.
4. What parts of the remix are learned by the LSTM, and what parts are produced by audio analysis or fixed rules? Give an example from your exported files.
5. Which version would you play in the AI Music Festival, and why?

**Optional advanced idea:** Transcribe a short single-instrument recording into MIDI using an external model, inspect the transcription for errors, then pass it to the existing personal-melody continuation step. Full commercial mixtures with many instruments are not reliable automatic MIDI inputs.

**Playing your remixes:** The preceding cell displays a player for each exported version. Run it, scroll to the players, and click **▶**. You can also find the saved WAV files in the final ZIP.


---
# Part 12 - Four-day submission and presentation checklist

**DAY 1:** Understand symbolic notes and tokenisation; build baseline; inspect LSTM gates and learn how training works.
**DAY 2:** Train model; conduct epochs, capacity/context and sampling experiments; save raw AI output; build a studio arrangement.
**DAY 3:** Create/upload MIDI primer; upload an instrumental or use the included fallback; analyse waveform/beat/chroma; compare three remixes.
**DAY 4:** Select the best composition; make evidence-based report; check reproducibility; prepare the five-minute group presentation.

### Submit three items
1. **Executed `.ipynb`** with your access number visible, completed outputs, graphs and explanations.
2. **Word report (6–8 pages)** covering: five member names/access numbers; musical representation; LSTM inputs/targets, embedding and memory; baseline and training results; controlled experiment table; personal motif; instrumental analysis; three remix comparisons; failure cases; attribution of learned versus rule-based stages; AI-use declaration.
3. **ZIP of audio/MIDI/figures**: raw LSTM melody, original built-in/uploaded excerpt (only if sharing allowed), three studio compositions, personal continuation, three uploaded-instrumental remixes, training graphs, MIDI files and a short README.

### Suggested marking (100)
Conceptual understanding 15; training and evaluation 15; three controlled experiments 20; original motif and AI continuation 10; instrumental analysis 10; remix investigation and final track 15; critical limitations and attribution 10; reproducibility and participation 5.

### Presentation (5 minutes per group)
Briefly explain the trained LSTM and chosen instrumental. Play 8 seconds original, 8 seconds raw LSTM, 10 seconds first remix, 15 seconds final remix. Use the remaining time to explain the greatest improvement or biggest failure, and which parts of the final track were produced using non-AI rules.

**Important:** Music is assessed primarily as an AI investigation, not a contest to produce commercially polished tracks. Work with audio you are permitted to share; do not include copyrighted commercial uploads in the submission package without permission.


In [ ]:
# FINAL BUNDLE: zip only the exported files from this group's notebook run.
import zipfile
archive=OUT.parent/f'{VARIANT_ID}_AI_Music_Studio_3_Outputs.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file() and f.name!='temporary_fullmix.wav':z.write(f,arcname=f.name)
print('Saved:',archive,'size MB:',round(archive.stat().st_size/1e6,1))
print('Files:',len([x for x in OUT.iterdir() if x.is_file()]))


In [ ]:
# Final export listing; on Colab download outputs before runtime disconnects.
print('FINAL OUTPUT FILES (including STUDIO WAV/MIDI):')
for f in sorted(OUT.iterdir()): print(f.name, f.stat().st_size,'bytes')
print(f'Group {GROUP_KEY} | variant {VARIANT_ID} | family {FAMILY} | assigned investigation {ASSIGNED_FACTOR}')
# Optional downloadable zip in Colab
import zipfile
zip_path=OUT.parent/f'{VARIANT_ID}_music_outputs.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():z.write(f,arcname=f.name)
print('ZIP:',zip_path)